# Run: analyse a player's game history and build their player model

**Milestone:** "Estimate player habits from game history". Input: a PGN file (e.g. a lichess export:
`https://lichess.org/api/games/user/<name>?evals=false&clocks=true`, downloaded by hand) and/or
Gambit's own logs (`logging.dir` in game.yaml). Output: `HabitProfile` table and a saved
`PlayerModel` JSON for the teacher.

Needs Stockfish (full strength analyst). Analysis takes `analysis_time_s` per position.

**TODO checklist**
- [ ] Put the PGN path and player name below (keep personal data out of git: `data/` is ignored?).
- [ ] Choose analysis depth/time and keep it fixed (ACPL depends on it; see player_model TODO-calibrate).
- [ ] Recency weighting for old games.

In [ ]:
%run ./player_model.ipynb

In [ ]:
from pathlib import Path
import pandas as pd

PGN_PATH = Path("../../data/history/player.pgn")     # TODO: your export
PLAYER = "your_lichess_name"                          # TODO
MAX_GAMES = 50
MODEL_OUT = Path("../../data/history") / f"{PLAYER}_model.json"

In [ ]:
games = []
with open(PGN_PATH) as f:
    while (g := chess.pgn.read_game(f)) is not None and len(games) < MAX_GAMES:
        if player_color_in_game(g, PLAYER) is not None:
            games.append(g)
print(len(games), "games for", PLAYER)

In [ ]:
assert find_engine_binary(), "install Stockfish for real analysis"
model = PlayerModel(PLAYER)
all_analyses = []
with Engine() as analyst:                              # full strength
    for i, g in enumerate(games):
        a = analyse_game(g, analyst, player_color_in_game(g, PLAYER), g.headers.get("Site", f"g{i}"))
        if a:
            model.update(a)
            all_analyses += a

In [ ]:
profile = build_habit_profile(PLAYER, all_analyses)
est = model.estimate()
print(f"Elo ~ {est.elo:.0f} [{est.elo_low:.0f}, {est.elo_high:.0f}] (placeholder formula), confidence {est.confidence:.2f}")
display(pd.DataFrame([w.__dict__ for w in est.weaknesses]))
display(pd.Series(profile.acpl_by_phase, name="ACPL by phase"))
model.save(MODEL_OUT)